# 📈 Chapter 2: Data and Sampling Distributions
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 2.1 Introduction
In the era of Big Data, it might seem like sampling is an outdated concept. If we have all the data, why sample? The reality is that data quality often matters more than data quantity. Furthermore, almost all statistical modeling and machine learning rely on the concept that the data you have is a *sample* from a larger true *population*.

This chapter explores how we draw samples, how statistics derived from those samples vary (Sampling Distributions), how to computationally estimate that variation (The Bootstrap), and the theoretical probability distributions that describe data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.utils import resample

# Set style for visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 2.")

## 2.2 Random Sampling and Sample Bias
A **sample** is a subset of data from a larger **population**. 
- **Simple Random Sample:** Every member of the population has an equal chance of being selected.
- **Stratified Sampling:** Dividing the population into strata (groups) and randomly sampling from each stratum to ensure representation.
- **Sample Bias:** Occurs when a sample is systematically different from the population (e.g., surveying only daytime shoppers to estimate total customer behavior).

### Selection Bias and Regression to the Mean
- **Data Snooping (Vast Search Effect):** If you search a massive dataset for patterns, you *will* find something that looks statistically significant purely by chance.
- **Regression to the Mean:** The phenomenon where extreme observations are usually followed by more central (average) observations. (e.g., The 'Sports Illustrated Cover Jinx').

## 2.3 Sampling Distribution of a Statistic
When we draw a sample and calculate a metric (like the mean), that metric is a **Statistic**. If we drew a *different* sample, we would get a *different* mean. 

The **Sampling Distribution** is the distribution of that statistic over many, many theoretical samples.

### The Central Limit Theorem (CLT)
The CLT is the most famous theorem in statistics. It states that the sampling distribution of the sample mean will be normally distributed, *regardless of the underlying distribution of the original data*, as long as the sample size is large enough (usually n > 30).

Let's prove the CLT computationally by generating a highly non-normal (right-skewed) population and taking samples from it.

In [ ]:
# 1. Create a heavily skewed population (e.g., Income data)
population = np.random.lognormal(mean=4.0, sigma=1.0, size=100000)

# 2. Define a function to draw samples and calculate the mean
def get_sample_means(pop, sample_size, num_samples=1000):
    sample_means = []
    for _ in range(num_samples):
        # Draw a simple random sample
        sample = np.random.choice(pop, size=sample_size, replace=False)
        sample_means.append(np.mean(sample))
    return sample_means

# 3. Get sampling distributions for different sample sizes (n)
means_n5 = get_sample_means(population, sample_size=5)
means_n20 = get_sample_means(population, sample_size=20)
means_n100 = get_sample_means(population, sample_size=100)

# 4. Plotting the results
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Original Population
sns.histplot(population[population < 500], bins=50, ax=axes[0, 0], color='gray')
axes[0, 0].set_title('Original Population (Highly Skewed)')

# Sample Size = 5
sns.histplot(means_n5, bins=50, ax=axes[0, 1], color='coral', kde=True)
axes[0, 1].set_title('Sampling Distribution of Mean (n=5)')

# Sample Size = 20
sns.histplot(means_n20, bins=50, ax=axes[1, 0], color='skyblue', kde=True)
axes[1, 0].set_title('Sampling Distribution of Mean (n=20)')

# Sample Size = 100
sns.histplot(means_n100, bins=50, ax=axes[1, 1], color='lightgreen', kde=True)
axes[1, 1].set_title('Sampling Distribution of Mean (n=100)')

plt.tight_layout()
plt.show()

print("Observation: As the sample size (n) increases, the distribution of the sample mean becomes perfectly bell-shaped (Normal), despite the original data being heavily skewed! This is the Central Limit Theorem in action.")

### Standard Error (SE)
The **Standard Error** is the standard deviation of the sampling distribution. It tells us how much our sample mean is likely to vary from the true population mean.

Formula: $SE = \frac{s}{\sqrt{n}}$ (where $s$ is the sample standard deviation and $n$ is sample size). 

Notice that to cut the standard error in half, you must multiply the sample size by four.

## 2.4 The Bootstrap
In the past, statisticians relied on mathematical formulas (like the SE formula above) to estimate how a statistic might vary. However, those formulas often assume normal distributions and only work for simple metrics like the mean.

What if we want to know the variability of the **Median**? Or the R-squared of a model? There are no simple formulas for those.

Enter **The Bootstrap**. It is a computationally intensive, highly effective technique. Instead of drawing from the theoretical population, we draw samples *from our existing sample*, **with replacement**, to simulate the sampling process. This allows us to estimate the standard error and confidence intervals of *any* statistic without mathematical formulas.

In [ ]:
# Let's say we only collected a single sample of 200 incomes
our_single_sample = np.random.choice(population, size=200, replace=False)

print(f"Our Sample Median: {np.median(our_single_sample):.2f}")

# We want to know how much this median might vary. We use the Bootstrap.
num_bootstraps = 2000
bootstrap_medians = []

for _ in range(num_bootstraps):
    # Draw a sample WITH replacement, same size as original sample
    boot_sample = resample(our_single_sample, replace=True, n_samples=len(our_single_sample))
    # Calculate the statistic of interest (Median)
    bootstrap_medians.append(np.median(boot_sample))

# Calculate the Bootstrap Standard Error
boot_se = np.std(bootstrap_medians)
print(f"Bootstrap Standard Error of the Median: {boot_se:.2f}")

# 2.5 Confidence Intervals
# A 90% Confidence Interval simply takes the 5th and 95th percentiles of the bootstrap distribution.
ci_lower = np.percentile(bootstrap_medians, 5)
ci_upper = np.percentile(bootstrap_medians, 95)

print(f"90% Confidence Interval for the Median: [{ci_lower:.2f}, {ci_upper:.2f}]")

# Visualize the Bootstrap Distribution
plt.figure(figsize=(7, 4))
sns.histplot(bootstrap_medians, bins=40, color='purple', alpha=0.6)
plt.axvline(ci_lower, color='red', linestyle='--', label='90% CI Bounds')
plt.axvline(ci_upper, color='red', linestyle='--')
plt.axvline(np.median(our_single_sample), color='black', label='Original Sample Median')
plt.title('Bootstrap Distribution of the Sample Median')
plt.xlabel('Median Value')
plt.legend()
plt.show()

## 2.6 Normal Distribution and QQ-Plots
The Normal (Gaussian) distribution is bell-shaped. It is standard if the mean is 0 and standard deviation is 1.

To visually check if a dataset follows a normal distribution, data scientists use a **QQ-Plot (Quantile-Quantile Plot)**. It plots the quantiles of the data against the theoretical quantiles of a normal distribution. If the data is normal, the points will fall perfectly on a diagonal line.

In [ ]:
# Generate Normal Data and Non-Normal (Heavy-Tailed) Data
normal_data = np.random.normal(0, 1, 1000)
t_data = np.random.standard_t(df=3, size=1000) # Heavy-tailed distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# QQ-Plot for Normal Data
stats.probplot(normal_data, dist="norm", plot=axes[0])
axes[0].set_title("QQ-Plot: Normal Data")

# QQ-Plot for Heavy-Tailed Data
stats.probplot(t_data, dist="norm", plot=axes[1])
axes[1].set_title("QQ-Plot: Heavy-Tailed Data (Student's t)")

plt.show()

print("Observation: In the heavy-tailed data QQ-plot, the points curve away from the line at the top right and bottom left. This indicates the data has more extreme outliers than a normal distribution would predict.")

## 2.7 Long-Tailed Distributions
While the normal distribution is mathematically convenient, real-world data (like stock returns, website traffic, or human wealth) is almost never perfectly normal. They often have **long tails** (highly skewed distributions). 

A "black swan" event in finance occurs precisely because practitioners falsely assume stock returns are normally distributed, drastically underestimating the probability of extreme events in the tail.

## 2.8 Student's t-Distribution
The t-distribution is similar to the normal distribution but has slightly thicker tails. It was developed by W.S. Gosset (under the pen name "Student") at the Guinness Brewery.

It is used instead of the normal distribution when calculating confidence intervals and t-tests, specifically when the **sample size is small (n < 30)** and the population standard deviation is unknown. As the sample size grows, the t-distribution becomes indistinguishable from the normal distribution.

## 2.9 Binomial Distribution
The Binomial distribution is used when we have a fixed number of trials ($n$), each trial has two possible outcomes (Success/Failure), and the probability of success ($p$) is constant.

Example: If 10% of users click an ad ($p=0.10$), what is the probability that exactly 3 users click the ad if we show it to 20 users ($n=20$)?

In [ ]:
n_trials = 20
p_success = 0.10

# Calculate the exact probability using the PMF (Probability Mass Function)
prob_3_clicks = stats.binom.pmf(k=3, n=n_trials, p=p_success)
print(f"Probability of exactly 3 clicks out of 20: {prob_3_clicks:.4f} (or {prob_3_clicks*100:.2f}%)")

# Let's visualize the entire Binomial distribution for this scenario
x = np.arange(0, 11)
pmf_values = stats.binom.pmf(x, n_trials, p_success)

plt.figure(figsize=(7, 4))
plt.bar(x, pmf_values, color='steelblue')
plt.title('Binomial Distribution (n=20, p=0.10)')
plt.xlabel('Number of Successes (Clicks)')
plt.ylabel('Probability')
plt.xticks(x)
plt.show()

## 2.10 Poisson and Related Distributions

While Binomial deals with a fixed number of trials, the **Poisson Distribution** models events that happen randomly over a fixed period of time or space. 
- **Lambda ($\lambda$):** The average rate of events per unit of time/space. (e.g., A call center receives an average of 5 calls per minute).

Closely related distributions:
- **Exponential Distribution:** Models the *time between* Poisson events (e.g., how long until the next call arrives?).
- **Weibull Distribution:** A generalization of the exponential distribution where the event rate can change over time (e.g., modeling mechanical failure rates where machines are more likely to fail as they age).

In [ ]:
# Example: Call center receiving an average of 5 calls per minute (Lambda = 5)
lam = 5

# 1. Poisson: What is the probability of receiving exactly 8 calls in a minute?
prob_8_calls = stats.poisson.pmf(k=8, mu=lam)
print(f"Probability of exactly 8 calls in a minute: {prob_8_calls:.4f}")

# 2. Exponential: Time between calls. 
# If lambda is 5 calls/min, average time between calls is 1/5 minutes (0.2 mins or 12 seconds).
# What is the probability that the next call arrives in less than 0.1 minutes?
# We use the CDF (Cumulative Distribution Function)
prob_less_than_01 = stats.expon.cdf(x=0.1, scale=1/lam)
print(f"Probability next call arrives in < 0.1 minutes: {prob_less_than_01:.4f}")

# Visualizing the Poisson Distribution
x_poisson = np.arange(0, 15)
poisson_pmf = stats.poisson.pmf(x_poisson, lam)

plt.figure(figsize=(7, 4))
plt.bar(x_poisson, poisson_pmf, color='seagreen')
plt.title('Poisson Distribution (Lambda = 5 calls/min)')
plt.xlabel('Number of Calls in One Minute')
plt.ylabel('Probability')
plt.xticks(x_poisson)
plt.show()

### Conclusion of Chapter 2
Understanding data and sampling distributions separates a statistically sound data scientist from someone who simply applies machine learning libraries blindly.

Key takeaways:
1. **The Bootstrap** is a modern, computationally powerful way to estimate error and confidence intervals without relying on strict mathematical assumptions.
2. **The Central Limit Theorem** guarantees that sampling distributions become normal as sample sizes increase, allowing us to make population estimates from samples.
3. Real data often has **long tails**; blindly assuming normality can lead to underestimating extreme risks.
4. Specific discrete distributions (**Binomial, Poisson**) are explicitly designed to model real-world business scenarios like click-through rates and queue/server loads.